In [1]:
from pyspark.sql import SparkSession
import os
import json

In [2]:
current_path = os.getcwd()

dir_path = os.path.dirname(os.path.dirname(current_path))

In [3]:
# Obter o diretório atual
current_path = os.getcwd()

# Verificar o ambiente
if "dev" in current_path:
    env = "dev"
elif "prod" in current_path:
    env = "prod"
else:
    env = "unknown"

print(f"A variável de ambiente é: {env}")

A variável de ambiente é: dev


In [4]:
with open(dir_path + f'/{env}-env/projeto_steam/config.json', 'r') as arquivo:
  config = json.load(arquivo)

In [5]:
jar_path = config['jar_path']

# Crie a sessão Spark com o JAR configurado
spark = SparkSession.builder \
    .appName("GoldStep") \
    .config("spark.jars", jar_path) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

24/11/15 19:31:49 WARN Utils: Your hostname, fececa-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
24/11/15 19:31:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


24/11/15 19:31:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [6]:
caminho_pasta = dir_path + f'/{env}-env/datalake/steam/wishlist/silver'

df = spark.read.format("parquet").load(caminho_pasta)

In [7]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- img: string (nullable = true)
 |-- name: string (nullable = true)
 |-- price: float (nullable = true)
 |-- short_description: string (nullable = true)
 |-- discount: long (nullable = true)
 |-- recommended: string (nullable = true)
 |-- minimum: string (nullable = true)
 |-- link: string (nullable = true)
 |-- file_date: date (nullable = true)



In [8]:
df.show()

+-------+--------------------+--------------------+------+--------------------+--------+--------------------+--------------------+--------------------+----------+
|     id|                 img|                name| price|   short_description|discount|         recommended|             minimum|                link| file_date|
+-------+--------------------+--------------------+------+--------------------+--------+--------------------+--------------------+--------------------+----------+
|1466860|https://shared.ak...|Age of Empires IV...| 99.99|Celebrating its f...|       0|Requires a 64-bit...|Requires a 64-bit...|https://store.ste...|2024-11-15|
|  15100|https://shared.ak...|Assassin's Creed™...| 59.99|Assassin's Creed™...|       0|                   -|  Supported OS: W...|https://store.ste...|2024-11-15|
| 201870|https://shared.ak...|Assassin's Creed®...| 59.99|Ezio Auditore wal...|       0| OS *: Windows® X...| OS *: Windows® X...|https://store.ste...|2024-11-15|
| 242050|https://share

In [9]:
url = config['url']

properties = {
    "user": config['user'],
    "password": config['password'],
    "driver": config['driver']
}

In [10]:
df.write \
.jdbc(url=url, table="wishlist", mode="overwrite", properties=properties)